In [ ]:
import pickle
import pandas as pd
from pathlib import Path
import numpy as np
import tensorflow as tf
import keras
from keras import layers
import math

In [ ]:
with Path("processed/token_to_id.pkl").open("rb") as f:
    token_to_id = pickle.load(f)
with Path("processed/id_to_token.pkl").open("rb") as f:
    id_to_token = pickle.load(f)

In [ ]:
with Path("processed/groups.csv").open("r") as f:
    first_line = f.readline()
CONTEXT_SIZE = first_line.count(",") // 2

In [ ]:
VOCAB_SIZE = len(id_to_token)
EMBEDDING_SIZE = 100
RANDOM_WORD_COUNT = 10

In [ ]:
def get_random(shape: tuple) -> int:
    return np.random.randint(0, len(id_to_token), shape)

In [ ]:
def count_lines(path: str, chunk_size: int = 1024 * 1024) -> int:
    count = 0

    with open(path, "rb") as file:
        while chunk := file.read(chunk_size):
            count += chunk.count(b"\n")

    return count

In [ ]:
DATA_LINE_COUNT = count_lines("processed/groups.csv")

In [ ]:
root_input = keras.Input((1,), name="root_input")
context_input = keras.Input(
    (CONTEXT_SIZE * 2 + RANDOM_WORD_COUNT,), name="context_input"
)

target_embedding = layers.Embedding(VOCAB_SIZE, EMBEDDING_SIZE, name="target_embedding")
context_embedding = layers.Embedding(
    VOCAB_SIZE, EMBEDDING_SIZE, name="context_embedding"
)

target_vec = target_embedding(root_input)  # (batch, 1, 300)
context_vec = context_embedding(context_input)  # (batch, 4, 300)

dots = layers.Dot(axes=(2, 2))([target_vec, context_vec])
logits = layers.Flatten()(dots)

output = layers.Activation("sigmoid")(logits)

model = keras.Model([root_input, context_input], outputs=output)
model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=["accuracy"],
)
model.summary()
keras.utils.plot_model(model, show_layer_names=True, show_shapes=True, dpi=100)

In [ ]:
GROUP_BATCH_SIZE = 10_000_000
EPOCHS = 4

for epoch in range(EPOCHS):
    group_batches = pd.read_csv(
        "processed/groups.csv",
        chunksize=GROUP_BATCH_SIZE,
        header=None,
    )
    for i, batch in enumerate(group_batches):
        print(
            f"Epoch {epoch + 1}/{EPOCHS}, Data Batch {i + 1}/{math.ceil(DATA_LINE_COUNT / GROUP_BATCH_SIZE)}"
        )

        batch_size = len(batch)

        root_words = batch[CONTEXT_SIZE]
        context_words = batch.drop(columns=[CONTEXT_SIZE])

        for i in range(RANDOM_WORD_COUNT):
            context_words[CONTEXT_SIZE * 2 + 1 + i] = get_random((batch_size,))

        labels = np.tile(
            np.array(
                [1 for _ in range(CONTEXT_SIZE * 2)]
                + [0 for _ in range(RANDOM_WORD_COUNT)]
            ),
            (batch_size, 1),
        )

        callbacks = [
            keras.callbacks.ModelCheckpoint(
                "best-model.keras",
                monitor="loss",
                mode="min",
                save_best_only=True,
                save_freq="epoch",
                verbose=1,
            ),
            # keras.callbacks.TensorBoard(update_freq="batch"),
        ]

        history = model.fit(
            [root_words, context_words],
            labels,
            batch_size=4096,
            epochs=1,
            callbacks=callbacks,
        )

        model.save("last-model.keras")
    batch

In [ ]:
batch